# Reconnaissance d'entités nommées

In [3]:
# Cellule 1 — installation éventuelle
# À lancer seulement si les bibliothèques ne sont pas déjà installées.

# !pip install -U agno lxml pydantic
# !pip install ollama
#!pip install ipywidgets

In [1]:
from pathlib import Path
from lxml import etree

from agno.agent import Agent
from agno.models.ollama import Ollama

from pydantic import BaseModel
from typing import List, Literal

import json
import re


In [2]:
DOSSIER_DATA = Path("../data")

FICHIER_TEI = DOSSIER_DATA / "Frêne_volume_1.xml"

FICHIER_JSON_LIGNES = DOSSIER_DATA / "lignes_tei_entites.json"

FICHIER_JSON_PROPOSITIONS = (
    DOSSIER_DATA / "propositions_entites_llm.json"
)

FICHIER_JSON_CORRIGE = (
    DOSSIER_DATA / "propositions_entites_corrigees.json"
)

FICHIER_JSON_PIVOT = (
    DOSSIER_DATA / "entites_pivot_offsets.json"
)

FICHIER_SORTIE = (
    DOSSIER_DATA / "Frêne_volume_1_entites_tei.xml"
)

MODELE_OLLAMA = "qwen3:8b"

NS_TEI = "http://www.tei-c.org/ns/1.0"
XML_NS = "http://www.w3.org/XML/1998/namespace"

NS = {"tei": NS_TEI}

XML_ID = f"{{{XML_NS}}}id"

In [3]:
def charger_tei(fichier):
    parser = etree.XMLParser(
        remove_blank_text=False
    )

    return etree.parse(
        str(fichier),
        parser
    )

arbre = charger_tei(FICHIER_TEI)
racine = arbre.getroot()

print("TEI chargé")

TEI chargé


In [4]:
def extraire_lignes_tei(arbre):
    lignes = arbre.xpath(
        "//tei:line",
        namespaces=NS
    )

    resultat = []

    for i, ligne in enumerate(lignes, start=1):

        xml_id = ligne.get(XML_ID)

        if xml_id is None:
            xml_id = f"ligne_{i:06d}"

        texte = "".join(
            ligne.itertext()
        ).strip()

        if texte:

            resultat.append({
                "xml_id": xml_id,
                "texte": texte
            })

    return resultat

In [5]:
lignes_json = extraire_lignes_tei(arbre)

with open(
    FICHIER_JSON_LIGNES,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        lignes_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"{len(lignes_json)} lignes extraites"
)

175 lignes extraites


In [6]:
class PropositionEntite(BaseModel):
    """Entité proposée par le LLM.

    On conserve uniquement la surface exacte et le type TEI.

    """

    surface: str
    type: Literal[
        "persName",
        "placeName",
        "orgName",
        "date",
        "roleName"
    ]


class ListeEntites(BaseModel):
    entites: List[PropositionEntite]


In [7]:
agent_entites = Agent(
    model=Ollama(id=MODELE_OLLAMA),
    output_schema=ListeEntites,
    instructions=[
        "Tu es spécialiste de l’annotation TEI et des textes manuscrits anciens en français.",
        "Tu travailles sur un journal manuscrit jurassien du XVIIIe siècle.",
        "Ta tâche est de repérer les entités nommées explicitement présentes dans le texte.",
        "Ne corrige pas le texte.",
        "Ne modernise pas l’orthographe.",
        "Ne reformule pas.",
        "La valeur surface doit être copiée exactement depuis le texte.",
        "Utilise uniquement les types TEI suivants : persName, placeName, orgName, date, roleName.",
        "persName désigne une personne, un prénom, un nom ou une désignation familiale individualisée.",
        "placeName désigne un lieu, village, ville, région, pays ou bâtiment localisé.",
        "orgName désigne une institution ou organisation.",
        "date désigne une date explicite, une année, un mois.",
        "roleName désigne une fonction ou un titre social associé à une personne.",
        "Si aucune entité n’est présente, retourne une liste vide.",
        "Retourne uniquement la structure demandée."
    ],
)


In [8]:
TYPES_TEI_AUTORISES = {
    "persName",
    "placeName",
    "orgName",
    "date",
    "roleName"
}


def nettoyer_entites(entites: list, texte: str) -> list:
    """Nettoie les propositions du LLM avant correction manuelle."""

    resultat = []
    deja_vu = set()

    for entite in entites:
        surface = str(entite.get("surface", "")).strip()
        type_tei = str(entite.get("type", "")).strip()

        if not surface:
            continue

        if type_tei not in TYPES_TEI_AUTORISES:
            continue

        # On élimine les hallucinations évidentes : la surface doit être dans la ligne.
        if surface not in texte:
            continue

        cle = (surface, type_tei)
        if cle in deja_vu:
            continue

        deja_vu.add(cle)
        resultat.append({
            "surface": surface,
            "type": type_tei
        })

    return resultat


def proposer_entites_ligne(ligne: dict) -> dict:
    """Demande au LLM de proposer des entités pour une ligne."""

    prompt = f"""
Analyse cette ligne issue d’un XML-TEI.

xml_id : {ligne["xml_id"]}
texte : {ligne["texte"]}

Repère les entités nommées présentes dans le texte.
"""

    try:
        reponse = agent_entites.run(prompt)
        contenu = reponse.content

        if isinstance(contenu, ListeEntites):
            entites = [e.model_dump() for e in contenu.entites]
        elif isinstance(contenu, dict):
            entites = contenu.get("entites", [])
        else:
            entites = []

        entites = nettoyer_entites(entites, ligne["texte"])

    except Exception as erreur:
        entites = []
        print(f"Erreur sur {ligne['xml_id']} : {erreur}")

    return {
        "xml_id": ligne["xml_id"],
        "texte": ligne["texte"],
        "entites_proposees": entites
    }


In [9]:
# Pour tester, commence avec une limite.
# Mets LIMITE_LIGNES = None pour traiter tout le fichier.
LIMITE_LIGNES = 10

lignes_a_traiter = (
    lignes_json
    if LIMITE_LIGNES is None
    else lignes_json[:LIMITE_LIGNES]
)

propositions = [
    proposer_entites_ligne(ligne)
    for ligne in lignes_a_traiter
]

with open(FICHIER_JSON_PROPOSITIONS, "w", encoding="utf-8") as f:
    json.dump(propositions, f, ensure_ascii=False, indent=2)

print(f"Propositions créées : {FICHIER_JSON_PROPOSITIONS}")


Propositions créées : ..\data\propositions_entites_llm.json


In [10]:
# Cellule de correction manuelle simple
# Principe : efface simplement, dans la liste ci-dessous, les entités fausses.
# Tu peux aussi corriger une surface ou un type directement.
# Ne garde que des dictionnaires de la forme :
# {"xml_id": "...", "surface": "...", "type": "..."}

with open(FICHIER_JSON_PROPOSITIONS, "r", encoding="utf-8") as f:
    propositions = json.load(f)

corrections_manuelles = [
    {
        "xml_id": ligne["xml_id"],
        "surface": entite["surface"],
        "type": entite["type"]
    }
    for ligne in propositions
    for entite in ligne.get("entites_proposees", [])
]

corrections_manuelles


[{'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_f93609b4-lineCount1-text',
  'surface': 'Chevres',
  'type': 'persName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_f93609b4-lineCount1-text',
  'surface': 'Pere',
  'type': 'roleName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_9d0b4f9b-lineCount2-text',
  'surface': 'M:',
  'type': 'roleName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_30cb9e7c-lineCount5-text',
  'surface': 'nusier',
  'type': 'persName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_30cb9e7c-lineCount5-text',
  'surface': 'Vauffelin',
  'type': 'placeName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_92f9ae1b-lineCount6-text',
  'surface': 'Magd',
  'type': 'persName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_92f9ae1b-lineCount6-text',
  'surface': 'cabinet',
  'type': 'placeName'},
 {'xml_id': 'f8-eSc_textblock_2873aa2a-eSc_line_12169d3c-lineCount7-text',
  'surface': 'Armoir',
  'type': 'persName'},
 {'xml_id': 'f8-eSc_textblock_287

In [14]:
# Validation de la liste corrigée et création du JSON intermédiaire
# À lancer après avoir supprimé/corrigé les mauvaises entrées dans corrections_manuelles.

with open(FICHIER_JSON_PROPOSITIONS, "r", encoding="utf-8") as f:
    propositions = json.load(f)

index_par_ligne = {ligne["xml_id"]: ligne for ligne in propositions}

for ligne in propositions:
    ligne["entites_proposees"] = []

for entite in corrections_manuelles:
    xml_id = entite.get("xml_id")
    surface = str(entite.get("surface", "")).strip()
    type_tei = str(entite.get("type", "")).strip()

    if xml_id not in index_par_ligne:
        print(f"xml_id ignoré : {xml_id}")
        continue

    if type_tei not in TYPES_TEI_AUTORISES:
        print(f"Type TEI ignoré pour {xml_id} : {type_tei}")
        continue

    if surface not in index_par_ligne[xml_id]["texte"]:
        print(f"Surface absente du texte pour {xml_id} : {surface}")
        continue

    index_par_ligne[xml_id]["entites_proposees"].append({
        "surface": surface,
        "type": type_tei
    })

propositions_corrigees = list(index_par_ligne.values())

with open(FICHIER_JSON_CORRIGE, "w", encoding="utf-8") as f:
    json.dump(propositions_corrigees, f, ensure_ascii=False, indent=2)

print(f"JSON corrigé enregistré : {FICHIER_JSON_CORRIGE}")


HTML(value='\n    <b>Ligne 1 / 8</b><br>\n    <b>xml_id :</b> f8-eSc_textblock_2873aa2a-eSc_line_f93609b4-line…

Textarea(value='[\n  {\n    "surface": "Chevres",\n    "type": "persName",\n    "confidence": "probable",\n   …

Output()

In [15]:
def trouver_occurrences(
    texte,
    surface
):

    return [
        {
            "start": m.start(),
            "end": m.end()
        }

        for m in re.finditer(
            re.escape(surface),
            texte
        )
    ]

In [16]:
def creer_pivot(
    donnees_corrigees
):

    compteur = 1

    resultat = []

    for ligne in donnees_corrigees:

        texte = ligne["texte"]

        entites = []

        for entite in ligne[
            "entites_proposees"
        ]:

            occurrences = (
                trouver_occurrences(
                    texte,
                    entite["surface"]
                )
            )

            for occ in occurrences:

                entites.append({

                    "entity_id":
                    f"ent_{compteur:06d}",

                    **entite,

                    "start":
                    occ["start"],

                    "end":
                    occ["end"]
                })

                compteur += 1

        resultat.append({

            "xml_id":
            ligne["xml_id"],

            "texte":
            texte,

            "entites":
            entites
        })

    return resultat

In [17]:
with open(
    FICHIER_JSON_CORRIGE,
    "r",
    encoding="utf-8"
) as f:

    corrige = json.load(f)

pivot = creer_pivot(corrige)

with open(
    FICHIER_JSON_PIVOT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pivot,
        f,
        ensure_ascii=False,
        indent=2
    )

print("JSON pivot créé")

JSON pivot créé


In [18]:
def creer_balise(
    entite,
    texte
):

    element = etree.Element(
        f"{{{NS_TEI}}}"
        f"{entite['type']}"
    )

    element.set(
        XML_ID,
        entite["entity_id"]
    )

    element.text = texte[
        entite["start"]:
        entite["end"]
    ]

    return element

In [19]:
arbre.write(
    str(FICHIER_SORTIE),
    encoding="UTF-8",
    xml_declaration=True,
    pretty_print=True
)

print(
    f"TEI enrichi créé : "
    f"{FICHIER_SORTIE}"
)

TEI enrichi créé : ..\data\Frêne_volume_1_entites_tei.xml
